# 02주차 실습: 파티셔닝과 저장 포맷

강의: [PDF](../pdf/week02_distributed_architecture.pdf) · **PDF 실제 페이지 5–18** (인쇄 번호와 다를 수 있음)

## 학습 목표
키 편향과 파일 포맷을 측정하여 레이크하우스 설계 근거를 만든다.

권장 구성: 개념 확인 10분 → 코드 실행 30분 → 변경 실험 30분 → 결과 토의 20분.
설치는 README.md를 따릅니다. 새 커널에서 위에서 아래로 실행하세요.

In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'labkit').is_dir()), None)
if root is None:
    candidate = here / '강의자료' / '실습코드'
    if candidate.is_dir(): root = candidate
if root is None:
    raise RuntimeError('실습코드 폴더에서 Jupyter를 시작하세요. README.md를 확인하세요.')
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labkit.common import ensure_data, DATA, output_dir, save_json
ensure_data()

## Range와 Hash 분할

In [ ]:
from labkit.common import stable_hash
df = pd.read_json(DATA / 'events.jsonl', lines=True)
df['range_partition'] = df.id // 600
df['product_partition'] = df.product_id.map(lambda x: stable_hash(x) % 4)
df['event_partition'] = df.event_id.map(lambda x: stable_hash(x) % 4)
counts = pd.DataFrame({c: df[c].value_counts().reindex(range(4), fill_value=0) for c in ['range_partition', 'product_partition', 'event_partition']})
display(counts)
counts.plot.bar(title='Rows per partition'); plt.tight_layout(); plt.show()
assert counts.sum().eq(len(df)).all()

## CSV·JSON·Parquet 비교

In [ ]:
import time
out = output_dir('week02')
base = df.drop(columns=['range_partition', 'product_partition', 'event_partition'])
base.to_csv(out / 'events.csv', index=False)
base.to_json(out / 'events.jsonl', orient='records', lines=True)
base.to_parquet(out / 'events.parquet', index=False, compression='snappy')
report = []
for name, read in [('events.csv', pd.read_csv), ('events.jsonl', lambda p: pd.read_json(p, lines=True)), ('events.parquet', pd.read_parquet)]:
    t = time.perf_counter(); loaded = read(out / name)
    report.append({'format': name, 'bytes': (out / name).stat().st_size, 'read_seconds': time.perf_counter() - t, 'rows': len(loaded)})
    assert len(loaded) == len(base)
display(pd.DataFrame(report))
pd.DataFrame(report).to_csv(out / 'formats.csv', index=False)

## 확장 과제 및 제출
동일 키의 Hash 분할로 편향이 해결되지 않는 이유를 설명하세요. 네트워크 분할 상황의 CP/AP 선택과 HDFS 복제 수, 날짜 파티셔닝 전략을 포함한 설계서를 제출하세요. 로컬 Parquet는 ACID 레이크하우스 구현이 아닙니다.

제출: 실행한 노트북, 결과 표/그림, 변경한 조건과 해석. outputs/weekXX에 저장된 자료를 첨부하세요.

평가 기준: 개념 연결 25%, 실행·재현성 30%, 비교 실험 25%, 해석·한계 20%.